# 故障予知① — ベースライン：固定閾値の再構成誤差をテストで測る

出典：第05章 02節／パス：`05_inspection/01_baseline.ipynb`／パッケージ：numpy・pandas・scikit-learn・matplotlib／実行時間の目安：約1分

このノート単体で実行できる。`input/sensors.csv`（合成のセンサーデータ）を読み込む。欠品時は内蔵生成へ切り替わる。
改善モデルのコードは `02_improved.ipynb` に分けてある。閾値は学習用正常の99%点に固定し（調整しない）、最終評価はテストセットで1回だけ行う。
正常の分け方（学習用・調整用・評価用）は改善モデルと同じにして、同じテストで比べられるようにしてある（調整用はこのノートでは使わない）。

In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RS = 0


def make_sensor_data(n_normal: int = 5000, n_abnormal: int = 200, seed: int = 5):
    r = np.random.default_rng(seed)
    base = r.normal(0, 1, size=(n_normal, 8))
    normal = base + 0.6 * base[:, [0]]  # センサー間の相関
    shift = np.array([2.5, -2.0, 1.5, 0.5, 0.0, 0.0, 1.0, -1.0])
    abnormal = base[:n_abnormal] * 1.5 + shift + r.normal(0, 0.5, size=(n_abnormal, 8))
    X = np.vstack([normal, abnormal])
    y = np.concatenate([np.zeros(n_normal, dtype=int), np.ones(n_abnormal, dtype=int)])
    return X, y


try:
    _s = pd.read_csv("input/sensors.csv")
    print("Inputを使用: input/sensors.csv")
    X = _s[[f"s{i}" for i in range(8)]].to_numpy()
    y = _s["label"].to_numpy()
except FileNotFoundError:
    X, y = make_sensor_data()
    print("Inputが無いため合成データを生成")
print(f"件数: {len(y)}, 異常率: {y.mean():.4f}")
# 正常は学習用・閾値調整用（500件）・評価用に分ける。調整用はPCAの学習に使わない
Xn_rest, Xn_te = train_test_split(X[y == 0], test_size=0.25, random_state=RS)
Xn_tr, Xn_tune = train_test_split(Xn_rest, test_size=500, random_state=RS)
# 異常は閾値調整用（3割＝60件）と評価用（7割＝140件）に分ける。学習には使わない
Xa_tune, Xa_te = train_test_split(X[y == 1], test_size=0.70, random_state=RS)
print(f"正常 学習:{len(Xn_tr)} 調整:{len(Xn_tune)} 評価:{len(Xn_te)} / 異常 調整:{len(Xa_tune)} 評価:{len(Xa_te)}")

Inputを使用: input/sensors.csv
件数: 5200, 異常率: 0.0385
正常 学習:3250 調整:500 評価:1250 / 異常 調整:60 評価:140


In [2]:
# 正常だけを3次元に圧縮するPCAを学び、戻せない度合いを異常スコアにする
scaler = StandardScaler().fit(Xn_tr)
pca = PCA(n_components=3, random_state=RS).fit(scaler.transform(Xn_tr))


def anomaly_score(Z: np.ndarray) -> np.ndarray:
    Zs = scaler.transform(Z)
    return ((Zs - pca.inverse_transform(pca.transform(Zs))) ** 2).sum(axis=1)


X_te = np.vstack([Xn_te, Xa_te])
y_te = np.concatenate([np.zeros(len(Xn_te), dtype=int), np.ones(len(Xa_te), dtype=int)])
s_te = anomaly_score(X_te)
print(f"[再構成誤差] テストPR-AUC={average_precision_score(y_te, s_te):.4f}（異常率 {y_te.mean():.3f}）")

[再構成誤差] テストPR-AUC=0.8465（異常率 0.101）


In [3]:
# ベースライン：閾値は学習用正常の99%点に固定する（異常例を見て決めない）
thr_fixed = float(np.quantile(anomaly_score(Xn_tr), 0.99))


def counts(thr: float) -> tuple:
    """テストでの拾えた件数・見逃し・誤検知・点検量（旗を立てた件数）"""
    flag = s_te >= thr
    tp = int((flag & (y_te == 1)).sum())
    fp = int((flag & (y_te == 0)).sum())
    return tp, int((y_te == 1).sum()) - tp, fp, tp + fp


tp, fn, fp, n_flag = counts(thr_fixed)
print(f"[ベースライン・固定閾値] 閾値{thr_fixed:.2f} / 適合率 {tp / n_flag:.3f} / 再現率 {tp / (tp + fn):.3f}（テスト）")
print(f"拾えた {tp}件 / 見逃し {fn}件 / 誤検知 {fp}件 / 点検量 {n_flag}件")
assert np.isfinite(s_te).all()

[ベースライン・固定閾値] 閾値9.80 / 適合率 0.892 / 再現率 0.707（テスト）
拾えた 99件 / 見逃し 41件 / 誤検知 12件 / 点検量 111件


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
plt.figure()
plt.hist(s_te[y_te == 0], bins=40, alpha=0.6, label="normal")
plt.hist(s_te[y_te == 1], bins=20, alpha=0.6, label="anomaly")
plt.axvline(thr_fixed, color="black", linestyle="--", label=f"fixed threshold ({thr_fixed:.2f})")
plt.xlabel("reconstruction error"); plt.ylabel("count"); plt.legend(); plt.title("score distributions (test, baseline)")
plt.savefig(f"{FIG}/code-result-j05-01.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j05-01.png")

saved code-result-j05-01.png


## 読み取りどころ

- 正常だけを学んでも異常を分離できること（PR-AUCが異常率を大きく上回る）。
- 固定閾値でも一定の検出ができること（改善幅は `02_improved.ipynb` で測る）。
- 固定閾値は点検量と見逃しの相談になっていないこと（改善モデルのファイルで基準を決めて選ぶ）。